In [1]:

import numpy as np
from scipy.optimize import brentq

# ==================================================
# 1. PARAMETER PENELITIAN
# ==================================================
b = 3.80
a = 0.18
d = 0.15
alpha = 0.15
m1 = 0.12
e1 = 0.60
d1 = 0.12
m2 = 0.10
e2 = 0.50
gamma = 1.00
d2 = 0.14

z_min, z_max = 1e-4, 1e3
root_tol = 1e-6

# Kesetimbangan yang dilaporkan dalam manuskrip
reported = np.array([
    [2.30691511, 8.87403103, 0.46097888],
    [11.28881211, 3.20416027, 6.92794472]
])

# ==================================================
# 2. PERSAMAAN KESETIMBANGAN
# ==================================================
def x_star(z):
    return (d1 + m2*z) / (e1*m1)

def y_star(z):
    return d2*(gamma + z) / (e2*m2*z)

def G(z):
    x = x_star(z)
    y = y_star(z)
    return b/(1 + alpha*y) - a*x - d - m1*y

# ==================================================
# 3. POLINOMIAL EKUIVALEN H(z) = 0
# ==================================================
C = d2/(e2*m2)
P = 1 + alpha*C
Q = alpha*C*gamma

L0 = a*d1/(e1*m1) + d + m1*C
L1 = a*m2/(e1*m1)

# H(z) = c3*z^3 + c2*z^2 + c1*z + c0
coeff = np.array([
    -L1*P,
    b - L0*P - L1*Q,
    -L0*Q - m1*C*gamma*P,
    -m1*C*gamma*Q
])

print("Koefisien H(z):")
print(coeff)

# ==================================================
# 4. SEMUA AKAR POLINOMIAL
# ==================================================
roots = np.roots(coeff)

print("\nSemua akar polinomial:")
for r in roots:
    print(f"{r.real:.12f} {r.imag:+.3e}i")

# Akar real, kemudian ambil yang positif
real_roots = sorted(
    float(r.real) for r in roots
    if abs(r.imag) < 1e-8
)

positive_roots = [z for z in real_roots if z > 0]

# ==================================================
# 5. PERIKSA AKAR BERULANG MELALUI TURUNAN
# ==================================================
critical_points = np.roots(np.polyder(coeff))

print("\nTitik kritis H(z):")
for r in critical_points:
    if abs(r.imag) < 1e-8:
        zc = float(r.real)
        print(
            f"z = {zc:.12f}, "
            f"H(z) = {np.polyval(coeff, zc):.6e}"
        )

# ==================================================
# 6. PEMBANDING: GRID LOGARITMIK + BRENT
# ==================================================
grid = np.geomspace(z_min, z_max, 20000)
brent_roots = []

for left, right in zip(grid[:-1], grid[1:]):
    f_left, f_right = G(left), G(right)

    if f_left * f_right < 0:
        z = brentq(G, left, right)
        brent_roots.append(z)

brent_roots = sorted(brent_roots)

print("\nAkar dari grid + Brent:")
for z in brent_roots:
    print(f"z = {z:.12f}")

# ==================================================
# 7. KELAYAKAN DAN RESIDUAL
# ==================================================
feasible = []

print("\nPemeriksaan akar positif:")
for z in positive_roots:
    x, y = x_star(z), y_star(z)
    residual_G = abs(G(z))
    residual_H = abs(np.polyval(coeff, z))

    is_feasible = (
        x > 0 and y > 0 and z > 0
        and residual_G < 1e-8
    )

    print(
        f"\nz = {z:.12f}\n"
        f"x = {x:.12f}\n"
        f"y = {y:.12f}\n"
        f"|G(z)| = {residual_G:.3e}\n"
        f"|H(z)| = {residual_H:.3e}\n"
        f"Feasible = {is_feasible}"
    )

    if is_feasible:
        feasible.append([x, y, z])

feasible = np.array(feasible)

# ==================================================
# 8. COCOKKAN DENGAN HASIL MANUSKRIP
# ==================================================
print("\nPerbandingan dengan manuskrip:")
for E in reported:
    distances = np.linalg.norm(feasible - E, axis=1)
    if len(distances) and np.min(distances) < 1e-5:
        print(f"{E}: MATCH")
    else:
        print(f"{E}: NOT MATCH")

print("\nJumlah akar real:", len(real_roots))
print("Jumlah akar real positif:", len(positive_roots))
print("Jumlah kesetimbangan layak:", len(feasible))
print("\nKesetimbangan layak (x, y, z):")
print(feasible)


Koefisien H(z):
[-0.355    2.57888 -0.80724 -0.14112]

Semua akar polinomial:
6.927944718532 +0.000e+00i
0.460978876536 +0.000e+00i
-0.124472890843 +0.000e+00i

Titik kritis H(z):
z = 4.681043445457, H(z) = 1.617601e+01
z = 0.161923690694, H(z) = -2.057221e-01

Akar dari grid + Brent:
z = 0.460978876536
z = 6.927944718532

Pemeriksaan akar positif:

z = 0.460978876536
x = 2.306915106300
y = 8.874031029447
|G(z)| = 4.441e-16
|H(z)| = 2.776e-17
Feasible = True

z = 6.927944718532
x = 11.288812109072
y = 3.204160268847
|G(z)| = 1.721e-15
|H(z)| = 9.792e-14
Feasible = True

Perbandingan dengan manuskrip:
[2.30691511 8.87403103 0.46097888]: MATCH
[11.28881211  3.20416027  6.92794472]: MATCH

Jumlah akar real: 3
Jumlah akar real positif: 2
Jumlah kesetimbangan layak: 2

Kesetimbangan layak (x, y, z):
[[ 2.30691511  8.87403103  0.46097888]
 [11.28881211  3.20416027  6.92794472]]
